# 📊 เส้นโค้ง Receiver Operating Characteristic (ROC)

ยินดีต้อนรับสู่สมุดโน้ตอธิบายการใช้งานจริงสำหรับ **ROC Curve**! ในสมุดโน้ตเล่มนี้ เราจะ:
1. กำหนดพิกัดทางคณิตศาสตร์ของ ROC Curve (True Positive Rate vs. False Positive Rate)
2. อิมพลีเมนต์การคำนวณ ROC จากศูนย์ (from scratch) โดยใช้ NumPy และตรวจสอบความถูกต้องกับ `scikit-learn`
3. จำลองผลการทำนายสำหรับ **Good Model** (การแยกคลาสได้ดี) และ **Poor Model** (คลาสมีความซ้อนทับกันสูง)
4. พล็อต ROC Curves ของทั้งสองโมเดลเพื่อแสดงให้เห็นภาพว่าระดับการแยกคลาสส่งผลต่อความโค้งเข้าหาการพล็อตมุมบนซ้ายอย่างไร
5. เปรียบเทียบระหว่าง ROC Curves กับ Precision-Recall (PR) Curves และอธิบายว่าทำไม PR curves จึงเป็นที่นิยมมากกว่าในระบบตรวจจับวัตถุอย่าง YOLO

เรามาเริ่มด้วยการนำเข้าไลบรารีที่จำเป็นกันเลยครับ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Set seed for reproducibility
np.random.seed(42)

## 1. การสร้างข้อมูลจำลอง (Data Generation)

เราจะจำลองตัวจำแนกประเภท (classifiers) สองรูปแบบดังนี้ครับ:
1.  **Good Classifier:** ทำนายคะแนนได้สูงสำหรับ Class 1 (ค่าเฉลี่ย 0.8) และคะแนนต่ำสำหรับ Class 0 (ค่าเฉลี่ย 0.2)
2.  **Poor Classifier:** ทำนายคะแนนที่มีความคาบเกี่ยวกันสูงมาก (ค่าเฉลี่ย 0.55 สำหรับ Class 1 และค่าเฉลี่ย 0.45 สำหรับ Class 0)

In [ ]:
n_samples = 150
y_true = np.concatenate([np.ones(75), np.zeros(75)]).astype(int)

# Good Classifier Scores
scores_good = np.concatenate([
    np.random.normal(0.8, 0.12, 75),
    np.random.normal(0.2, 0.12, 75)
])
scores_good = np.clip(scores_good, 0.0, 1.0)

# Poor Classifier Scores
scores_poor = np.concatenate([
    np.random.normal(0.55, 0.2, 75),
    np.random.normal(0.45, 0.2, 75)
])
scores_poor = np.clip(scores_poor, 0.0, 1.0)

## 2. การคำนวณเส้นโค้ง ROC จากศูนย์ (from Scratch)

เรามาเขียนฟังก์ชันเพื่อคำนวณ TPR และ FPR ณ ทุกระดับเกณฑ์ (threshold) ที่เป็นไปได้กันครับ
ขั้นตอนการดำเนินงาน:
1. เรียงลำดับเกณฑ์ (thresholds) จากมากไปน้อย (เริ่มจาก $1.0$ ลงไปจนถึง $0.0$)
2. ในแต่ละเกณฑ์ คำนวณ y_pred = (scores $\ge$ threshold)
3. คำนวณค่า TP, TN, FP, FN
4. คำนวณ $TPR = TP / (TP + FN)$ และ $FPR = FP / (TN + FP)$

In [ ]:
def custom_roc_curve(y_true, scores):
    """
    Calculate ROC curve coordinates (FPR, TPR) from scratch.
    """
    thresholds = np.sort(scores)[::-1]
    thresholds = np.concatenate([[1.001], thresholds])
    
    tprs = []
    fprs = []
    
    for thresh in thresholds:
        y_pred = (scores >= thresh).astype(int)
        
        TP = np.sum((y_true == 1) & (y_pred == 1))
        TN = np.sum((y_true == 0) & (y_pred == 0))
        FP = np.sum((y_true == 0) & (y_pred == 1))
        FN = np.sum((y_true == 1) & (y_pred == 0))
        
        tpr = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        fpr = FP / (TN + FP) if (TN + FP) > 0 else 0.0
        
        tprs.append(tpr)
        fprs.append(fpr)
        
    return np.array(fprs), np.array(tprs), thresholds

# Calculate custom ROC curves
fpr_good, tpr_good, thresh_good = custom_roc_curve(y_true, scores_good)
fpr_poor, tpr_poor, thresh_poor = custom_roc_curve(y_true, scores_poor)

# Verify against sklearn
fpr_sk, tpr_sk, _ = roc_curve(y_true, scores_good)
print("FPR curves close?", np.allclose(np.interp(fpr_sk, fpr_good, fpr_good), fpr_sk))

## 3. การแสดงผลเส้นโค้ง ROC (Visualizing the ROC Curves)

เรามาพล็อตเส้นโค้ง ROC ของทั้งสองโมเดลเพื่อเปรียบเทียบความแตกต่างที่เกิดขึ้นกันครับ

In [ ]:
plt.figure(figsize=(10, 7))

plt.plot(fpr_good, tpr_good, color='forestgreen', linewidth=3, label='Good Classifier')
plt.plot(fpr_poor, tpr_poor, color='darkorange', linewidth=2.5, linestyle='-.', label='Poor Classifier')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Guess (AUC = 0.50)')

# Annotate specific threshold points on the Good Classifier Curve
indices_to_label = [len(thresh_good)//4, len(thresh_good)//2, 3*len(thresh_good)//4]
for idx in indices_to_label:
    t = thresh_good[idx]
    f = fpr_good[idx]
    p = tpr_good[idx]
    plt.scatter(f, p, color='red', s=60, zorder=5)
    plt.annotate(f"Thresh={t:.2f}", (f+0.02, p-0.03), fontsize=10)

plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR / Recall)')
plt.title('ROC Curves: Good vs. Poor Separation')
plt.xlim(-0.02, 1.02)
plt.ylim(-0.02, 1.02)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='lower right')
plt.show()

## 💡 ROC Curve vs. PR Curve (ทำความเข้าใจแนวคิด)
ทำไมในวงการคอมพิวเตอร์วิชัน (Computer Vision) จึงนิยมใช้ **Precision-Recall (PR) Curves** มากกว่า **ROC Curves** สำหรับงานตรวจจับวัตถุ (Object Detection) ครับ?
1.  **ความอ่อนไหวต่อปัญหาคลาสไม่สมดุล (Sensitivity to Class Imbalance):** เส้นโค้ง ROC จะไม่มีความอ่อนไหวต่อปัญหาคลาสไม่สมดุล (class imbalance) เนื่องจากสูตรของ False Positive Rate ($FPR = FP / (TN + FP)$) มีค่า True Negatives ($TN$) รวมอยู่ในตัวหารด้วย ในงานตรวจจับวัตถุ พื้นหลังทั้งหมดที่มองว่าเป็นวัตถุอื่นที่ไม่สนใจ ($TN$) นั้นมีขนาดเป็นอนันต์ ดังนั้น แม้ว่าโมเดลจะทำนายผิดพลาดว่าเป็นวัตถุเป้าหมาย (เกิด false alarms หรือ $FP$) เป็นพันๆ ครั้ง ตัวหารก็ยังมีขนาดใหญ่มากจนส่งผลให้ $FPR \approx 0$ ซึ่งทำให้เส้นโค้ง ROC ดูดีจนเกือบสมบูรณ์แบบ ทั้งๆ ที่ความเป็นจริงโมเดลทำงานได้แย่มากครับ!
2.  **พฤติกรรมของเส้นโค้ง PR (PR Curve Behavior):** สูตรการคำนวณ Precision ($TP / (TP + FP)$) จะไม่นำค่า True Negatives มาเกี่ยวข้องเลย ดังนั้น หากโมเดลทำนายผิดพลาดบ่อยครั้ง (false alarms) ค่า Precision จะตกลงทันที ทำให้สะท้อนประสิทธิภาพที่แท้จริงของการตรวจจับภายใต้สภาวะที่คลาสไม่สมดุลได้อย่างแม่นยำครับ